## Task 4 (LS2): 

Implement a program which 
(a) given one of the feature models, 
(b) a user specified value of k, 

Reports the top-k latent semantics extracted using CP-decomposition of a three modal(image-feature-label) tensor under the selected features space. Each latent semantic should be presented in the form of a list of label-weight pairs, ordered in decreasing order of weights.

– Store the latent semantics in a properly named output file <br>
– List label-weight pairs, ordered in decreasing order of weights

In [116]:
IMAGE_INPUT = input(
"""
Provide one of the following:
1. An Image ID in the Caltech101 dataset in range [0, 8676].
2. The name of an image file in /Code/input/ directory (eg: image.jpg).
"""
)

FEATURE_SPACE = input("Provide a feature space [color, hog, avgpool, layer3, fc].")

K = int(input("Enter K (K should be less than the selected rank which is 5), the number latent features to be found."))

In [117]:
import tensorlearn as tl
import numpy as np
from utils.database_utils import retrieve

In [118]:
feature_vectors = retrieve(f'{FEATURE_SPACE}.pt')

In [119]:
def find_centers(feature_vectors):
    cluster_centers = {}
    for i in range(len(feature_vectors)):
        if i % 2 == 0:
            if feature_vectors[i][0] in cluster_centers:
                cluster_centers[feature_vectors[i][0]].append(feature_vectors[i][1])
            else:
                cluster_centers[feature_vectors[i][0]] = []
    for label in cluster_centers:
        cluster_centers[label] = np.average(cluster_centers[label], axis=0)
    return cluster_centers


In [122]:
def find_cp_decomposition(feature_vectors, K):
    def create_feature_format(feature_vectors):
        # For feature space of every image, find distance from cluster centers
        centers = find_centers(feature_vectors)
        result = []
        for i in range(len(feature_vectors)): #loop over all images
            # find distance from every center in find_centers(feature_vectors)
            distance = []
            if i % 2 == 0:
                # distance = len(labels) x len(feature vector) array
                for label in centers:
                    distance.append(np.abs(feature_vectors[i][1] - centers[label]))
                result.append(distance)
        print(np.shape(result))
        return np.array(result)    

    def reduce_factor_matrix(factor, K):
        result = np.zeros((factor.shape[0], K))
        for i in range(factor.shape[0]):
            result[i] = factor[i][:K]
        return result

    tensor = create_feature_format(feature_vectors)
    weights, factors = tl.cp_als_rand_init(tensor, 5, 50)
    
    new_weights = weights[0:K]
    new_factors = []

    new_factors.append(reduce_factor_matrix(factors[0], K))
    new_factors.append(reduce_factor_matrix(factors[1], K))
    new_factors.append(reduce_factor_matrix(factors[2], K))
    return new_weights, new_factors, tensor

weights, factors, tensor = find_cp_decomposition(feature_vectors, K)
    # print(new_factors)

(2170, 31, 900)


In [124]:
tensor_hat=tl.cp_to_tensor(weights, factors)
error=tensor_hat-tensor
error_ratio=tl.tensor_frobenius_norm(error)/tl.tensor_frobenius_norm(tensor)
print(error_ratio)

1.242326543433525
